In [13]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn tensorflow boto3

mkdir: cannot create directory ‘/home/ec2-user/tmpdir’: File exists


In [2]:
# Prepare dataset
%cd ~/SageMaker/
! rm -rf dataset/ dataset.zip
!wget -nc -q -O "dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/ayush1220/cifar10"

!mkdir dataset -p
!unzip -q -o ./dataset.zip -d ./dataset/
!mv ./dataset/cifar10/* ./dataset/
!rm -r ./dataset/cifar10 dataset.zip

%cd ./dataset/
!zip -rq ../dataset.zip .

%cd ..

/home/ec2-user/SageMaker
/home/ec2-user/SageMaker/dataset
/home/ec2-user/SageMaker


In [3]:
# Import pip packages
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.model_selection import train_test_split
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.s3 import S3Uploader

I0000 00:00:1782863112.061799    5240 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [4]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [5]:
# Upload dataset zip file to S3
inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [43]:
# Find mean, std
_dataset = keras.utils.image_dataset_from_directory(
    './dataset/train',
    image_size=(32, 32),
    batch_size=64,
    shuffle=False,
)

mean    = tf.zeros(3)
sq_mean = tf.zeros(3)
n_pixels = 0

for images, _ in _dataset:
    images = tf.cast(images, tf.float32) / 255.0
    b, h, w, c = images.shape
    n_pixels += b * h * w
    mean     += tf.reduce_sum(images,       axis=[0, 1, 2])
    sq_mean  += tf.reduce_sum(images ** 2,  axis=[0, 1, 2])

mean  /= tf.cast(n_pixels, tf.float32)
std    = tf.sqrt(sq_mean / tf.cast(n_pixels, tf.float32) - mean ** 2)

mean, std

Found 50000 files belonging to 10 classes.


(<tf.Tensor: shape=(3,), dtype=float32, numpy=array([0.49139827, 0.48215774, 0.44653437], dtype=float32)>,
 <tf.Tensor: shape=(3,), dtype=float32, numpy=array([0.24703138, 0.24348348, 0.26157853], dtype=float32)>)

In [21]:
from sagemaker.tensorflow import TensorFlow

estimator = TensorFlow(
    entry_point="train.py",
    framework_version="2.19.0",
    py_version="py312",
    instance_type="ml.m6i.xlarge",
    instance_count=1,
    output_path="s3://{}/{}/output/".format(bucket, prefix),
    role=role,
    hyperparameters = {
        'epochs': 20,
        'batch_size': 64,
        'lr': 0.001,
    },
    metric_definitions = [
        {"Name": "train:loss", "Regex": r"loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"accuracy: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"val_loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"val_accuracy: ([0-9\.]+)"},
    ],
)

estimator.fit({"train": f's3://{bucket}/{prefix}/dataset.zip'})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: tensorflow-training-2026-07-01-02-44-50-359


2026-07-01 02:44:54 Starting - Starting the training job...
2026-07-01 02:45:08 Starting - Preparing the instances for training...
2026-07-01 02:45:52 Downloading - Downloading the training image...
2026-07-01 02:46:22 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-07-01 02:46:36.464755: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-01 02:46:36.496217: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the ap

In [ ]:
# Inference src source dir
!mkdir src
!cp train.py src/inference.py
!touch src/requirements.txt
!echo "numpy" >> src/requirements.txt
!echo "Pillow" >> src/requirements.txt
!echo "tensorflow" >> src/requirements.txt

In [ ]:
# Deploy: Create Model + Endpoint Configuration + Endpoint
predictor = estimator.deploy(
    initial_instnace_count = 1,
    instance_type = "ml.g5.xlarge",
    entry_point = "inference.py",
    source_dir = "src"
)

In [ ]:
# Create Model
from sagemaker.tensorflow import TensorFlowModel

model = TensorFlowModel(
    name = f'cifar10-model-{int(time.time())}',
    model_data = estimator.model_data,
    role = role,
    entry_point = "inference.py",
    framework_version = "2.19.0",
    source_dir = "src"
)

predictor = model.deploy(
    initial_instance_count = 1,
    instance_type = "ml.g5.xlarge"
)

In [38]:
# Create new model + Update Endpoint
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model(
    name=f'cifar10-model-{int(time.time())}',
    entry_point = "inference.py",
    source_dir = "src"
)
model_name = model.name
model._create_sagemaker_model(instance_type="ml.g5.xlarge")

new_config = f'endpoint-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.g5.xlarge"
    }]
)

INFO:sagemaker.tensorflow.model:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating model with name: cifar10-model-1782884332


{'EndpointConfigArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint-config/endpoint-configuration-1782884334',
 'ResponseMetadata': {'RequestId': '9f72f818-9640-453a-a1b3-961d2819b9ff',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '9f72f818-9640-453a-a1b3-961d2819b9ff',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '114',
   'date': 'Wed, 01 Jul 2026 05:38:54 GMT'},
  'RetryAttempts': 0}}

In [39]:
sm.create_endpoint(
    EndpointName = "cifar10-endpoint-4",
    EndpointConfigName = new_config
)

# sm.update_endpoint(
#     EndpointName = "cifar10-endpoint",
#     EndpointConfigName = new_config
# )

{'EndpointArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint/cifar10-endpoint-4',
 'ResponseMetadata': {'RequestId': '04fb46b7-0011-4fdb-9f33-539d7e0a047f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '04fb46b7-0011-4fdb-9f33-539d7e0a047f',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '86',
   'date': 'Wed, 01 Jul 2026 05:38:57 GMT'},
  'RetryAttempts': 0}}

In [47]:
CLASSES = _dataset.class_names
CLASSES

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

In [ ]:
# Inference test
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'cifar10-endpoint-4'
image_path = "0006.png"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 1, 'confidence': 0.9999881061024473}
automobile
